# MODUL PRAKTIKUM BIG DATA
## Pertemuan 9 — Basis Data NoSQL: Instalasi MongoDB & Integrasi dengan PySpark

| | |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |
| **Pertemuan** | 9 |
| **Topik** | Konsep NoSQL, Instalasi MongoDB, Operasi CRUD, Integrasi MongoDB-PySpark |
| **Estimasi Waktu** | 150 menit |
| **Prasyarat** | Modul Pertemuan 1-7 selesai, **UTS (Pertemuan 8) telah dilaksanakan** |

---

### Cara Menggunakan Modul Ini

Seperti pola sebelumnya: **Sub-bab 9.1 – 9.3** (instalasi MongoDB) dikerjakan di **terminal Ubuntu**. Mulai **Sub-bab 9.4**, seluruh instruksi dikerjakan di **Jupyter Notebook**.

> **Konsistensi versi:** Modul ini menggunakan **MongoDB Community Server 8.0** (rilis stabil, tersedia resmi untuk Ubuntu 22.04 & 24.04) dan **PyMongo** (driver Python resmi MongoDB) versi terbaru. Berbeda dengan pair Spark/PySpark yang **wajib** versi identik, PyMongo memiliki kompatibilitas mundur yang luas (mendukung MongoDB Server 4.0 hingga 8.0 sekaligus, serta Python 3.9-3.13) — sehingga **tidak perlu mengunci versi PyMongo secara presisi**, cukup pasang versi terbaru. Environment `bigdata` (Python 3.11) tetap dipakai, tidak ada environment baru.


---
## Recap & Selamat Melewati UTS!

Selamat, anda baru saja menyelesaikan UTS yang mencakup Pertemuan 1-7. Mulai pertemuan ini kita membuka topik baru dalam ekosistem Big Data: **penyimpanan data NoSQL**.

- [ ] VM Ubuntu menyala, `conda activate bigdata` berhasil
- [ ] Hadoop aktif jika dibutuhkan (`start-dfs.sh`, `start-yarn.sh`) — pertemuan ini **tidak wajib** menyalakan Hadoop di awal, namun akan kita gunakan kembali di bagian akhir

## Tujuan Pembelajaran

Setelah menyelesaikan Pertemuan 9, mahasiswa mampu:
1. Menjelaskan konsep basis data NoSQL dan perbedaannya dengan basis data relasional.
2. Menginstall MongoDB Community Server di Ubuntu.
3. Melakukan operasi CRUD (*Create, Read, Update, Delete*) menggunakan `mongosh` dan PyMongo.
4. Mengintegrasikan data dari MongoDB ke dalam Spark DataFrame untuk dianalisis lebih lanjut.

---

## 9.1 Mengapa NoSQL? Mengingat Kembali Konsep *Variety* dari Big Data

Ingat kembali kerangka **5V** pada Pertemuan 3 — salah satunya adalah **Variety** (keragaman format data). Sejauh ini, seluruh data yang kita olah (CSV, Parquet) bersifat **terstruktur**, mengikuti skema tabel yang kaku (baris & kolom tetap). Di dunia nyata, banyak data justru **semi-terstruktur** dan skemanya dapat bervariasi antar-baris — misalnya data profil pengguna aplikasi, di mana sebagian pengguna memiliki data alamat lengkap sementara yang lain tidak, atau setiap produk pada katalog e-commerce memiliki atribut yang sama sekali berbeda (baju punya atribut "ukuran", elektronik punya atribut "voltase").

| Aspek | Basis Data Relasional (SQL) | Basis Data NoSQL (mis. MongoDB) |
|---|---|---|
| **Struktur data** | Tabel dengan skema tetap (kolom harus didefinisikan di awal) | Dokumen (mirip JSON) dengan skema fleksibel |
| **Relasi antar data** | Menggunakan JOIN antar tabel | Data terkait dapat disematkan (*embedded*) langsung dalam satu dokumen |
| **Skalabilitas** | Umumnya vertikal (tambah kapasitas 1 server) | Dirancang untuk skalabilitas horizontal (tambah banyak server) |
| **Kapan digunakan** | Data terstruktur, butuh konsistensi transaksi ketat | Data semi-terstruktur, skema sering berubah, butuh skala sangat besar |

>**Bukan pengganti, tapi pelengkap.** Dalam ekosistem Big Data nyata, HDFS/Spark (data terstruktur skala besar) dan MongoDB (data semi-terstruktur, akses cepat per-dokumen) sering dipakai **bersamaan**, masing-masing untuk kebutuhan yang paling sesuai — bukan saling menggantikan.

**MongoDB** adalah basis data NoSQL bertipe *document store* paling populer, menyimpan data dalam format **BSON** (Binary JSON) yang terorganisir dalam **collection** (setara "tabel") berisi **document** (setara "baris", namun berbentuk seperti objek JSON).


---
## 9.2 Instalasi MongoDB Community Server 8.0

**Jalankan seluruh langkah berikut di Terminal Ubuntu.**

**Langkah 1: Impor kunci GPG resmi MongoDB**

```bash
curl -fsSL https://pgp.mongodb.com/server-8.0.asc | \
   sudo gpg -o /usr/share/keyrings/mongodb-server-8.0.gpg --dearmor
```

**Langkah 2: Tambahkan repository MongoDB sesuai versi Ubuntu kalian**

Untuk **Ubuntu 22.04 (Jammy)**:
```bash
echo "deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-8.0.gpg ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/8.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-8.0.list
```

Untuk **Ubuntu 24.04 (Noble)**:
```bash
echo "deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-8.0.gpg ] https://repo.mongodb.org/apt/ubuntu noble/mongodb-org/8.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-8.0.list
```

>Perhatikan kata `jammy` (untuk 22.04) atau `noble` (untuk 24.04) pada perintah di atas — gunakan sesuai versi Ubuntu anda dari Pertemuan 1. Jalankan **hanya salah satu**, sesuai versi kalian.

**Langkah 3: Perbarui daftar paket & instal MongoDB**

```bash
sudo apt update
sudo apt install -y mongodb-org
```

**Langkah 4: Jalankan MongoDB sebagai service & aktifkan agar otomatis menyala setiap boot**

```bash
sudo systemctl start mongod
sudo systemctl enable mongod
```

**Langkah 5: Verifikasi MongoDB berjalan**

```bash
sudo systemctl status mongod
```

Cari baris `Active: active (running)` berwarna hijau sebagai tanda keberhasilan. Tekan `q` untuk keluar dari tampilan status.

Verifikasi versi yang terinstall:

```bash
mongod --version
```

---

## 9.3 Operasi Dasar dengan `mongosh` (MongoDB Shell)

`mongosh` adalah *command-line interface* resmi untuk berinteraksi langsung dengan MongoDB — mirip peran `hdfs dfs` untuk HDFS atau `psql` untuk PostgreSQL.

**Masuk ke MongoDB shell:**

```bash
mongosh
```

Anda akan masuk ke prompt interaktif `test>`. Coba jalankan perintah-perintah dasar berikut **di dalam `mongosh`**:

```javascript
// Berpindah/membuat database bernama "praktikum_bigdata"
use praktikum_bigdata

// Menyisipkan satu dokumen ke collection "produk" (otomatis dibuat jika belum ada)
db.produk.insertOne({ nama_produk: "Kopi Susu", kategori: "Minuman", harga: 18000, stok: 120 })

// Menampilkan seluruh dokumen pada collection "produk"
db.produk.find()

// Menampilkan dengan format lebih rapi
db.produk.find().pretty()
```

Keluar dari `mongosh` dengan mengetik:

```javascript
exit
```

>**Perhatikan:** kita tidak perlu mendefinisikan struktur/skema `produk` terlebih dahulu seperti pada tabel SQL — MongoDB otomatis membuat collection saat dokumen pertama disisipkan. Inilah wujud nyata dari *flexible schema* yang dibahas pada Sub-bab 9.1.


---
## Melanjutkan di Jupyter Notebook

Install driver PyMongo di environment `bigdata`:

```bash
conda activate bigdata
pip install pymongo
```

Verifikasi:

```bash
python -c "import pymongo; print('PyMongo', pymongo.version)"
```

Lalu jalankan seperti biasa:

```bash
cd ~/praktikum-bigdata
jupyter notebook
```

## 9.4 Koneksi ke MongoDB dari Python

In [1]:
from pymongo import MongoClient

# Membuat koneksi ke MongoDB yang berjalan di localhost, port default 27017
client = MongoClient("mongodb://localhost:27017/")

# Memilih (atau otomatis membuat) database
db = client["praktikum_bigdata"]

# Memilih (atau otomatis membuat) collection
koleksi_produk = db["produk"]

print("Koneksi ke MongoDB berhasil!")
print("Daftar database yang ada:", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database yang ada: ['admin', 'config', 'local', 'praktikum_bigdata']


Jika muncul daftar nama database tanpa error, koneksi PyMongo ke MongoDB anda telah berhasil.

---

## 9.5 Operasi CRUD dengan PyMongo

### 9.5.1 Create — Menyisipkan Dokumen

In [2]:
# Menghapus data lama jika ada (agar percobaan kita mulai dari kondisi bersih)
koleksi_produk.delete_many({})

# insert_one() — menyisipkan satu dokumen
koleksi_produk.insert_one({
    "nama_produk": "Kopi Susu",
    "kategori": "Minuman",
    "harga": 18000,
    "stok": 120,
    "tags": ["best-seller", "dingin"]
})

# insert_many() — menyisipkan banyak dokumen sekaligus
koleksi_produk.insert_many([
    {"nama_produk": "Roti Bakar", "kategori": "Makanan", "harga": 15000, "stok": 45, "tags": ["hangat"]},
    {"nama_produk": "Es Teh", "kategori": "Minuman", "harga": 8000, "stok": 200, "tags": ["dingin", "murah"]},
    {"nama_produk": "Kentang Goreng", "kategori": "Makanan", "harga": 20000, "stok": 60, "tags": ["gurih"]},
    {"nama_produk": "Cappuccino", "kategori": "Minuman", "harga": 25000, "stok": 80, "tags": ["hangat", "best-seller"]},
])

print("Jumlah dokumen di collection 'produk':", koleksi_produk.count_documents({}))

Jumlah dokumen di collection 'produk': 5


> **Perhatikan** setiap produk memiliki jumlah `tags` yang berbeda-beda — sebagian punya 1 tag, sebagian punya 2. Ini **tidak mungkin dilakukan secara alami** pada tabel SQL biasa tanpa tabel relasi tambahan, namun sangat wajar pada model dokumen MongoDB.

### 9.5.2 Read — Membaca Dokumen

In [3]:
# find_one() — mengambil satu dokumen pertama yang cocok
satu_produk = koleksi_produk.find_one({"kategori": "Minuman"})
print("Satu produk kategori Minuman:", satu_produk)

print()

# find() — mengambil SEMUA dokumen yang cocok (mengembalikan cursor, di-looping atau di-list())
print("Seluruh produk kategori Minuman:")
for p in koleksi_produk.find({"kategori": "Minuman"}):
    print(f"  - {p['nama_produk']} (Rp{p['harga']:,})")

print()

# Filter dengan operator perbandingan: harga lebih dari 15000
print("Produk dengan harga > 15000:")
for p in koleksi_produk.find({"harga": {"$gt": 15000}}):
    print(f"  - {p['nama_produk']} (Rp{p['harga']:,})")

Satu produk kategori Minuman: {'_id': ObjectId('6ae2e3b41a9d2b06d28718b3'), 'nama_produk': 'Kopi Susu', 'kategori': 'Minuman', 'harga': 18000, 'stok': 120, 'tags': ['best-seller', 'dingin']}

Seluruh produk kategori Minuman:
  - Kopi Susu (Rp18,000)
  - Es Teh (Rp8,000)
  - Cappuccino (Rp25,000)

Produk dengan harga > 15000:
  - Kopi Susu (Rp18,000)
  - Kentang Goreng (Rp20,000)
  - Cappuccino (Rp25,000)


### 9.5.3 Update — Memperbarui Dokumen

In [4]:
# update_one() — memperbarui SATU dokumen pertama yang cocok
koleksi_produk.update_one(
    {"nama_produk": "Es Teh"},
    {"$set": {"harga": 9000}}   # menaikkan harga Es Teh
)

# update_many() — memperbarui SEMUA dokumen yang cocok
# Contoh: mengurangi stok seluruh produk kategori Minuman sebanyak 5 (simulasi penjualan)
koleksi_produk.update_many(
    {"kategori": "Minuman"},
    {"$inc": {"stok": -5}}   # $inc mengubah nilai relatif (increment/decrement)
)

print("Setelah update:")
for p in koleksi_produk.find({}, {"_id": 0, "nama_produk": 1, "harga": 1, "stok": 1}):
    print(f"  - {p['nama_produk']}: harga=Rp{p['harga']:,}, stok={p['stok']}")

Setelah update:
  - Kopi Susu: harga=Rp18,000, stok=115
  - Roti Bakar: harga=Rp15,000, stok=45
  - Es Teh: harga=Rp9,000, stok=195
  - Kentang Goreng: harga=Rp20,000, stok=60
  - Cappuccino: harga=Rp25,000, stok=75


### 9.5.4 Delete — Menghapus Dokumen

In [5]:
# delete_one() — menghapus SATU dokumen pertama yang cocok
koleksi_produk.delete_one({"nama_produk": "Kentang Goreng"})

print("Jumlah dokumen tersisa:", koleksi_produk.count_documents({}))
print("Daftar produk tersisa:", [p["nama_produk"] for p in koleksi_produk.find({}, {"nama_produk": 1, "_id": 0})])

Jumlah dokumen tersisa: 4
Daftar produk tersisa: ['Kopi Susu', 'Roti Bakar', 'Es Teh', 'Cappuccino']


---
## 9.6 Mengintegrasikan Data MongoDB ke Spark DataFrame

Setelah data tersimpan di MongoDB, kita sering perlu menganalisisnya menggunakan kemampuan Spark yang sudah dipelajari sejak Pertemuan 4-7 (agregasi skala besar, join dengan sumber data lain, dsb.). Cara paling sederhana dan tanpa perlu instalasi tambahan: **ambil data dari MongoDB dengan PyMongo, ubah menjadi pandas DataFrame, lalu konversi menjadi Spark DataFrame** — memanfaatkan kembali seluruh skill Pertemuan 2 (pandas) dan Pertemuan 4 (Spark) sekaligus.

In [6]:
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder \
    .appName("Pertemuan9-MongoDBSpark") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Langkah 1: Ambil seluruh dokumen dari MongoDB sebagai list of dict
dokumen_mongo = list(koleksi_produk.find({}))

# Langkah 2: Ubah menjadi pandas DataFrame
df_pandas = pd.DataFrame(dokumen_mongo)

# Langkah 3: Tipe data khusus MongoDB (_id, dan kolom berisi list seperti "tags")
# perlu disesuaikan dulu agar Spark dapat membacanya dengan benar
df_pandas["_id"] = df_pandas["_id"].astype(str)
df_pandas["tags"] = df_pandas["tags"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")

# Langkah 4: Konversi ke Spark DataFrame
df_spark_produk = spark.createDataFrame(df_pandas)

df_spark_produk.printSchema()
df_spark_produk.show()

26/10/29 13:13:04 WARN Utils: Your hostname, rindani-Latitude-3410 resolves to a loopback address: 127.0.1.1; using 10.90.70.182 instead (on interface wlp0s20f3)
26/10/29 13:13:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/29 13:13:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- _id: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- stok: long (nullable = true)
 |-- tags: string (nullable = true)



+--------------------+-----------+--------+-----+----+-------------------+
|                 _id|nama_produk|kategori|harga|stok|               tags|
+--------------------+-----------+--------+-----+----+-------------------+
|6ae2e3b41a9d2b06d...|  Kopi Susu| Minuman|18000| 115|best-seller, dingin|
|6ae2e3b41a9d2b06d...| Roti Bakar| Makanan|15000|  45|             hangat|
|6ae2e3b41a9d2b06d...|     Es Teh| Minuman| 9000| 195|      dingin, murah|
|6ae2e3b41a9d2b06d...| Cappuccino| Minuman|25000|  75|hangat, best-seller|
+--------------------+-----------+--------+-----+----+-------------------+



Sekarang data yang tadinya tersimpan di MongoDB sudah menjadi Spark DataFrame biasa — dapat langsung diolah menggunakan seluruh teknik yang telah dipelajari (`groupBy`, `join`, window function, disimpan sebagai Parquet ke HDFS, dan seterusnya).

In [7]:
from pyspark.sql.functions import avg, count

# Contoh analisis sederhana: rata-rata harga & jumlah produk per kategori
df_spark_produk.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga"),
    count("nama_produk").alias("jumlah_produk")
).show()

spark.stop()
print("SparkSession ditutup.")

+--------+------------------+-------------+
|kategori|   rata_rata_harga|jumlah_produk|
+--------+------------------+-------------+
| Minuman|17333.333333333332|            3|
| Makanan|           15000.0|            1|
+--------+------------------+-------------+

SparkSession ditutup.


> **Untuk skala produksi:** pendekatan "PyMongo → pandas → Spark" di atas cocok untuk data berukuran sedang (ribuan-jutaan dokumen). Untuk integrasi skala sangat besar (miliaran dokumen), tim data engineering biasanya menggunakan **MongoDB Spark Connector** resmi yang memungkinkan Spark membaca MongoDB secara terdistribusi langsung — namun membutuhkan instalasi & konfigurasi tambahan yang di luar cakupan praktikum ini.


---
## Latihan Mandiri

Pastikan `client`, `db`, dan `koleksi_produk` dari Sub-bab 9.4 masih aktif sebelum mengerjakan latihan berikut (jalankan ulang cellnya jika kernel sudah di-restart).

**Soal 1.** Sisipkan (`insert_many`) lima dokumen baru ke collection `produk` dengan kategori bebas (boleh kategori baru selain Makanan/Minuman, misalnya "Snack").

In [9]:
# Jawaban Soal 1 di sini
koleksi_produk.insert_many([
    {"nama": "Keripik Singkong", "kategori": "Snack", "harga": 12000, "stok": 50, "tags":["gurih"]},
    {"nama": "Biskuit Coklat", "kategori": "Snack", "harga": 8000, "stok": 100, "tags": ["manis", "best-seller"]},
    {"nama": "Kacang Atom", "kategori": "Snack", "harga": 10000, "stok": 70, "tags": ["gurih"]},
    {"nama": "Perment kopi", "kategori": "Snack", "harga": 5000, "stok": 100, "tags": ["murah"]},
    {"nama": "Biskuit Keju", "kategori": "Snack", "harga": 8000, "stok": 100, "tags":["manis", "gurih"]},
])
print("Jumlah dokumen di collection 'produk' : ", koleksi_produk.count_documents({}))

Jumlah dokumen di collection 'produk' :  14


**Soal 2.** Gunakan `find()` dengan operator `$lt` (kurang dari) untuk menampilkan seluruh produk dengan `harga` di bawah 20000.

In [10]:
# Jawaban Soal 2 di sini
hasil = koleksi_produk.find({"harga": {"$lt": 20000}})

for doc in hasil:
    print(doc)

{'_id': ObjectId('6ae2e3b41a9d2b06d28718b3'), 'nama_produk': 'Kopi Susu', 'kategori': 'Minuman', 'harga': 18000, 'stok': 115, 'tags': ['best-seller', 'dingin']}
{'_id': ObjectId('6ae2e3b41a9d2b06d28718b4'), 'nama_produk': 'Roti Bakar', 'kategori': 'Makanan', 'harga': 15000, 'stok': 45, 'tags': ['hangat']}
{'_id': ObjectId('6ae2e3b41a9d2b06d28718b5'), 'nama_produk': 'Es Teh', 'kategori': 'Minuman', 'harga': 9000, 'stok': 195, 'tags': ['dingin', 'murah']}
{'_id': ObjectId('6ae2ebe91a9d2b06d28718b8'), 'nama': 'Keripik Singkong', 'kategori': 'Snack', 'harga': 12000, 'stok': 50, 'tags': ['gurih']}
{'_id': ObjectId('6ae2ebe91a9d2b06d28718b9'), 'nama': 'Biskuit Coklat', 'kategori': 'Snack', 'harga': 8000, 'stok': 100, 'tags': ['manis', 'best-seller']}
{'_id': ObjectId('6ae2ebe91a9d2b06d28718ba'), 'nama': 'Kacang Atom', 'kategori': 'Snack', 'harga': 10000, 'stok': 70, 'tags': ['gurih']}
{'_id': ObjectId('6ae2ebe91a9d2b06d28718bb'), 'nama': 'Perment kopi', 'kategori': 'Snack', 'harga': 5000, 's

**Soal 3.** Gunakan `update_many()` untuk menambahkan field baru bernama `diskon` bernilai `0` ke **seluruh** dokumen pada collection `produk` (petunjuk: gunakan operator `$set`).

In [12]:
# Jawaban Soal 3 di sini
hasil_update = koleksi_produk.update_many(
    {},
    {"$set":{"diskon": 0}}
)

print(f"Jumlah dokumen yang dimodifikasi: {hasil_update.modified_count}")


Jumlah dokumen yang dimodifikasi: 14


**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: berikan **satu contoh skenario nyata** (boleh di luar konteks e-commerce yang sering kita pakai) di mana MongoDB akan menjadi pilihan lebih tepat dibanding tabel relasional biasa, dan jelaskan alasannya berdasarkan konsep *flexible schema*. Tulis jawaban pada markdown cell di bawah ini.

*(MongoDB bisa digunakan pada aplikasi IoT / Tracker Kendaraan yang mencatat data sensor secara real-time. Setiap jenis kendaraan atau sensor dapat mengirimkan struktur data yang berbeda-beda(misalnya GPS, suhu mesin, atau status baterai) tanpa perlu mengubah struktur basis data terlebih dahulu. Sifat flexibel schema MongoDB memudahkan sistem menerima data heterogen ini secara dinamis dan efisien)*


---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 10 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Platform "Kopi Nusantara" (skenario yang sama dari Tugas Mandiri Pertemuan 7) ingin mulai menyimpan **katalog menu** mereka di MongoDB — karena setiap cabang punya menu musiman/spesial dengan atribut yang berbeda-beda (tidak semua menu punya atribut yang sama, misalnya hanya menu kopi yang punya atribut "tingkat_kepahitan", sementara menu makanan punya atribut "pedas_level"). Kalian ditugaskan membangun dan menganalisis katalog ini.

### Instruksi Pengerjaan

Buat notebook baru **`Tugas9_[NIM]_[Nama Lengkap].ipynb`**, koneksikan ke MongoDB, lalu kerjakan bagian **A sampai E** berikut:

---

**A. Membangun Katalog (Create)** *(bobot 20%)*

Buat database baru bernama `kopi_nusantara`, collection bernama `menu`. Sisipkan **minimal 12 dokumen** menu dengan field bervariasi sesuai kategorinya:
- Menu kategori **"Kopi"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `tingkat_kepahitan` (skala 1-5).
- Menu kategori **"Makanan"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `pedas_level` (skala 0-3).
- Boleh menambahkan field lain sesuai kreativitas anda (mis. `stok`, `rekomendasi_barista`).

**B. Kueri Bertingkat (Read)** *(bobot 20%)*

Tampilkan hasil dari **tiga** kueri berikut :
1. Seluruh menu kategori "Kopi" dengan `tingkat_kepahitan` di atas 3.
2. Seluruh menu dengan `harga` di antara 15000 dan 30000.
3. Menu yang **tidak memiliki** field `pedas_level` sama sekali.

**C. Pembaruan Data (Update)** *(bobot 15%)*

Tambahkan field `status` bernilai `"Tersedia"` ke seluruh dokumen menggunakan `update_many()`. Kemudian, ubah `status` menjadi `"Habis"` khusus untuk 2 menu pilihan anda menggunakan `update_one()`.

**D. Integrasi ke Spark** *(bobot 25%)*

Ambil seluruh data dari collection `menu`, ubah menjadi Spark DataFrame (ikuti pola Sub-bab 9.6 — perhatikan field yang mungkin kosong/`null` pada sebagian dokumen, seperti `tingkat_kepahitan` yang hanya dimiliki menu Kopi). Tampilkan:
1. `printSchema()` hasil konversi.
2. Rata-rata `harga` per `kategori` menggunakan `groupBy`.

**E. Kesimpulan** *(bobot 20%)*

Tulis pada markdown cell (**minimal 100 kata**): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas9_[NPM]_[Nama Lengkap].ipynb` melalui asisten praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan MongoDB (`sudo systemctl status mongod`) aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Create | Minimal 12 dokumen tersimpan dengan field sesuai ketentuan kategori | 20% |
| B. Read | Ketiga kueri menghasilkan output yang tepat sesuai instruksi | 20% |
| C. Update | Field `status` berhasil ditambahkan & diperbarui sesuai instruksi | 15% |
| D. Integrasi Spark | Konversi ke Spark DataFrame berhasil; analisis `groupBy` benar | 25% |
| E. Kesimpulan | Refleksi menunjukkan pemahaman konseptual yang tepat tentang tantangan skema | 20% |
